# Lab 22-07 · Converter Loops

**Predict first**, then run. The handout is [README.md](README.md).

In [1]:
import collections
import math
import re
import shutil
import subprocess
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import control as ct

L, C, R, VG = 0.6e-6, 62.5e-6, 0.12, 12.0
FC, PM = 60e3, 60.0


def ngspice(netlist):
    if shutil.which("ngspice") is None:
        print(f"ngspice is not installed: skipping {netlist} (the README lists the expected values)")
        return collections.defaultdict(lambda: float("nan"))
    out = subprocess.run(["ngspice", "-b", netlist], capture_output=True, text=True).stdout
    return {mm.group(1): float(mm.group(2)) for mm in re.finditer(r"^(\w+)\s*=\s*([-+0-9.eE]+)", out, re.M)}

## Part 1 · The type-III buck loop

*Predict* the crossover and phase margin of Example 22-07.3, and what a ±50% change in C3 does.

In [2]:
ph = math.degrees(np.angle(ct.buck_gvd(FC, VG, L, C, R)))
b, k = ct.kfactor(PM, ph)
fz, fp = FC / k, FC * k
wi = 1 / abs(ct.type3(FC, 1.0, fz, fp) * ct.buck_gvd(FC, VG, L, C, R))
c1, c2, r2, r3, c3 = ct.type3_components(10e3, wi, k, fz, fp)
print(f"boost {b:.1f} deg, k {k:.2f}, fz {fz/1e3:.2f} kHz, fp {fp/1e3:.1f} kHz")
print(f"C1 {c1*1e9:.3f} nF, C2 {c2*1e9:.4f} nF, R2 {r2:.0f} ohm, R3 {r3:.0f} ohm, C3 {c3*1e9:.3f} nF")


def net(f, r1, r2, c1, c2, r3, c3):
    s = 2j * math.pi * f
    zf = (r2 + 1 / (s * c1)) * (1 / (s * c2)) / (r2 + 1 / (s * c1) + 1 / (s * c2))
    zi = r1 * (r3 + 1 / (s * c3)) / (r1 + r3 + 1 / (s * c3))
    return zf / zi


for scale in (0.5, 1.0, 1.5):
    loop = lambda f, scale=scale: net(f, 10e3, r2, c1, c2, r3, c3 * scale) * ct.buck_gvd(f, VG, L, C, R)
    fc, pm = ct.crossover_and_pm(loop)
    print(f"C3 x {scale}: crossover {fc/1e3:.1f} kHz, phase margin {pm:.1f} deg")
s = ngspice("loop.cir")
print({kk: s[kk] for kk in ("vout", "fcross", "pm")})

boost 126.5 deg, k 4.20, fz 14.27 kHz, fp 252.2 kHz
C1 11.237 nF, C2 0.6741 nF, R2 992 ohm, R3 600 ohm, C3 1.052 nF
C3 x 0.5: crossover 41.9 kHz, phase margin 62.2 deg
C3 x 1.0: crossover 60.0 kHz, phase margin 60.0 deg
C3 x 1.5: crossover 76.7 kHz, phase margin 48.3 deg
{'vout': 1.2, 'fcross': 60000.83, 'pm': 60.0012}


## Part 2 · The boost's RHP zero

*Predict* |G_vd| at 1 kHz and the magnitude and phase at f_RHP = 275 kHz.

In [3]:
f = np.logspace(1, math.log10(275.0197e3), 2000)
g = ct.boost_gvd(f, 5.0, 0.28, 1e-6, 22e-6, 10 / 3)
print(f"|G| at 1 kHz {abs(ct.boost_gvd(1e3, 5.0, 0.28, 1e-6, 22e-6, 10/3)):.3f}; at f_RHP {abs(g[-1]):.4f}, phase {np.degrees(np.unwrap(np.angle(g)))[-1]:.1f} deg")
s = ngspice("boost_avg.cir")
print({kk: s[kk] for kk in ("vout", "g1k", "gz", "phz")})

|G| at 1 kHz 6.956; at f_RHP 0.0781, phase -224.5 deg
{'vout': 5.0, 'g1k': 6.956099, 'gz': 0.07811501, 'phz': -224.5443}


## Part 3 · Slope compensation

*Predict* how an initial 0.1 A error evolves over 10 cycles for m_a = 0, m2/2 and m2 (Example 22-07.4).

In [4]:
m1, m2 = (5 - 3.3) / 2.2, 3.3 / 2.2
plt.figure()
for ma, lab in ((0.0, "m_a = 0"), (m2 / 2, "m_a = m2/2"), (m2, "m_a = m2")):
    a = ct.perturbation_ratio(m1, m2, ma)
    err = [0.1 * a ** n for n in range(11)]
    plt.plot(range(11), err, marker="o", label=f"{lab} (alpha = {a:.2f})")
plt.yscale("symlog", linthresh=1e-3); plt.xlabel("cycle"); plt.ylabel("current error (A)"); plt.legend(); plt.show()

## Part 4 · Closed-loop output impedance

*Predict* the peak of |Z_out/(1 + T)| for Part 1's loop against 1/(2 pi f_c C).

In [5]:
loop = lambda f: net(f, 10e3, r2, c1, c2, r3, c3) * ct.buck_gvd(f, VG, L, C, R)
f = np.logspace(2, 6, 3000)
zo = np.abs(ct.buck_zout(f, L, C, R))
zc = np.abs(ct.buck_zout(f, L, C, R) / (1 + np.array([loop(x) for x in f])))
plt.figure()
plt.loglog(f, zo * 1e3, label="open loop"); plt.loglog(f, zc * 1e3, label="closed loop")
plt.axhline(1e3 / (2 * math.pi * FC * C), color="0.4", ls="--", label="1/(2 pi f_c C)")
plt.xlabel("frequency (Hz)"); plt.ylabel("|Z_out| (mOhm)"); plt.legend(); plt.show()
print(f"closed-loop peak {zc.max()*1e3:.1f} mOhm at {f[zc.argmax()]/1e3:.1f} kHz; estimate {1e3/(2*math.pi*FC*C):.1f} mOhm")

closed-loop peak 47.9 mOhm at 58.6 kHz; estimate 42.4 mOhm


## Discussion

Compare your predictions with the results, then answer the discussion questions in the README.